# **Introducción al análisis de datos en Python** 
#### Profesor: Santiago Neira


## Pandas Avanzado: Pegues y Agrupaciones de Datos

### Unir bases de datos (`merge`)

Ya aprendimos cómo concatenar filas y columnas de diferentes bases de datos. Para hacer esto es necesario que la cantidad de columnas y filas, respectivamente, de los dataframes a juntar sean los mismos y que sus índices o llaves también lo sean.

No obstante, muchas veces cuando trate de juntar bases de datos, notará que no necesariamente todas las llaves están presentes en ambas bases de datos, o que incluso, a cada fila de la base izquierda, querrá pegarle más de una fila de la base derecha, o viceversa.

A la hora de hacer pegues más complejos, hablamos de que vamos a utilizar un `merge`. 

Comencemos con la sintaxis del `merge`. Para pegar dos bases de datos, usted usará un comando similar al siguiente:

```python
pd.merge(left = left_dataframe, right = right_dataframe, on = "alguna(s)_columa(s)", how = "left|right|inner|outer")
```

Los argumentos que toma la función son:
- `left`: dataframe que va de primero.
- `right`: dataframe que va de segundo.
- `on`: es la columna o la lista de columnas que determinan qué filas de una tabla coinciden con qué filas de la segunda tabla. Comúnmente a estas variables se les llaman las llaves del pegue y debe identificar a cada observación de forma única. A veces, las columnas que desea fusionar tienen nombres diferentes en los datos. Por ejemplo, suponga que tiene dos bases de datos, una que registra el dinero mensual gastado por persona en almacenes Éxito y otra que tiene características personales de las personas. Usted podría tratar de juntar ambas bases con el identificador de fila o persona de cada base que en este caso podría ser la cédula, sin embargo, en un dataframe tal vez la variable se llame "cc" mientras que en el otro puede que se llame "cédula". En esos casos, puede especificar los nombres de columna por separado para cada marco de datos utilizando los argumentos "left_on" y "right_on".
- `how`: es el método a usar, por defecto Pandas usa el método "inner". Más adelante exploraremos más al respecto.

<center>
<div>
<img src="./img/merges.png" width="400"/>
</div>
</center>

Tenemos cuatro grandes métodos para relacionar las bases porque no siempre tenemos una coincidencia uno a uno (one to one) entre las filas. Estos cuatro métodos afectan la forma en que Pandas trata los datos no coincidentes y eso es lo que veremos más adelante.

<center>
<div>
<img src="./img/one-many.png" width="400"/>
</div>
</center>



In [1]:
import pandas as pd
import numpy as np

In [2]:
# ejemplos de pegues
left_dataframe = pd.DataFrame({"ID": [1,2,3,4], "left_side": "Izquierda"})
right_dataframe = pd.DataFrame({"ID": [3,4,5,6], "right_side": "Derecha"})

In [3]:
left_dataframe

,ID,left_side
0,1,Izquierda
1,2,Izquierda
2,3,Izquierda
3,4,Izquierda


In [4]:
right_dataframe

,ID,right_side
0,3,Derecha
1,4,Derecha
2,5,Derecha
3,6,Derecha


#### Left merge
En un Left merge lo que más nos interesa son los datos del lado IZQUIERDO a los cuales queremos pegarles columnas de una base de datos en el lado DERECHO.

Para hacer eso, cortamos las filas en el marco de datos DERECHO y pegamos partes en el marco de datos IZQUIERDO. Recuerde, nos preocupamos principalmente por el lado IZQUIERDO y solo queremos datos del lado DERECHO si tiene alguna de las mismas ID. Entonces, si algo en el marco de datos DERECHO no coincide o no existe, entonces tenemos que hacer cosas para mantener las columnas de la misma longitud. Lo hacemos agregando NaN para llenar el vacío o descartando algunas filas por completo.

En este ejemplo, el lado IZQUIERDO tiene los ID 1, 2, 3 y 4:
- El lado DERECHO no tiene ID 1 o 2, por lo que agregamos NaN porque necesitamos que las columnas tengan la misma longitud.
- El lado DERECHO tiene datos para los ID 3 y 4, así que lo agregamos como una nueva columna.
- El lado IZQUIERDO no tiene ID 5 o 6, por lo que no necesitamos esa información del DERECHO y se descarta.

<center>
<div>
<img src="./img/left_merge.png" width="400"/>
</div>
</center>

In [5]:
# Left merge con "ID" como llave
pd.merge(left = left_dataframe, right = right_dataframe, on = "ID", how = "left")

,ID,left_side,right_side
0,1,Izquierda,NaN
1,2,Izquierda,NaN
2,3,Izquierda,Derecha
3,4,Izquierda,Derecha


#### Right merge
Los Right merges funcionan igual que los Left merges, la diferencia es que nos preocupamos principalmente por el lado DERECHO y nos gustaría agregar datos desde el IZQUIERDO si tienen ID coincidentes.

<center>
<div>
<img src="./img/right_merge.png" width="400"/>
</div>
</center>

In [7]:
# Right merge con "ID" como llave
pd.merge(left = left_dataframe, right = right_dataframe, on = "ID", how = "right")

,ID,left_side,right_side
0,3,Izquierda,Derecha
1,4,Izquierda,Derecha
2,5,NaN,Derecha
3,6,NaN,Derecha


#### Inner merge
Con un Inner merge, cortamos ambos marcos de datos y solo pegamos las cosas que coinciden. Si una ID no está en ambos marcos de datos, no la mantenemos y no agregamos NaN.

<center>
<img src="./img/inner_merge.png" width="400"/>
</center>

In [8]:
# Inner merge con "ID" como llave
pd.merge(left = left_dataframe, right = right_dataframe, on = "ID", how = "inner")

,ID,left_side,right_side
0,3,Izquierda,Derecha
1,4,Izquierda,Derecha


#### Outer merge
Con un Outer merge, cortamos ambos marcos de datos y mantenemos todo de ambos lados. Luego agregamos NaN para llenar los espacios en blanco.

<center>
<img src="./img/outer_merge.png" width="400"/>
</center>

In [9]:
# Outer merge con "ID" como llave
pd.merge(left = left_dataframe, right = right_dataframe, on = "ID", how = "outer")

,ID,left_side,right_side
0,1,Izquierda,NaN
1,2,Izquierda,NaN
2,3,Izquierda,Derecha
3,4,Izquierda,Derecha
4,5,NaN,Derecha
5,6,NaN,Derecha


In [11]:
# Creamos dos DataFrames: empleados y departamentos
empleados = pd.DataFrame({
    'id': [1, 2, 3, 4],
    'nombre': ['Ana', 'Luis', 'Carlos', 'Sofía'],
    'departamento_id': [10, 20, 10, 30]
})

departamentos = pd.DataFrame({
    'departamento_id': [10, 20, 30],
    'departamento': ['Ventas', 'Marketing', 'TI']
})


In [12]:
empleados

,id,nombre,departamento_id
0,1,Ana,10
1,2,Luis,20
2,3,Carlos,10
3,4,Sofía,30


In [13]:
departamentos

,departamento_id,departamento
0,10,Ventas
1,20,Marketing
2,30,TI


In [17]:
df_unido=pd.merge(empleados, departamentos, on='departamento_id',how='left')
df_unido

,id,nombre,departamento_id,departamento
0,1,Ana,10,Ventas
1,2,Luis,20,Marketing
2,3,Carlos,10,Ventas
3,4,Sofía,30,TI


In [19]:

# 🔗 Unimos usando la columna en común
### base_izquierda.merge(base_derecha, on='columna(s)_comun(es)', how='tipo_de_union')
df_unido = empleados.merge(departamentos, on='departamento_id',how='left')
print(df_unido)

   id  nombre  departamento_id departamento
0   1     Ana               10       Ventas
1   2    Luis               20    Marketing
2   3  Carlos               10       Ventas
3   4   Sofía               30           TI


In [21]:
# Queremos saber si el nombre de la persona empieza con vocal
def empieza_con_vocal(nombre):
    return nombre[0].lower() in 'aeiou'

empleados['empieza_vocal'] = empleados['nombre'].apply(empieza_con_vocal)
print(empleados)

   id  nombre  departamento_id  empieza_vocal
0   1     Ana               10           True
1   2    Luis               20          False
2   3  Carlos               10          False
3   4   Sofía               30          False


### Groupby


Uno de los métodos más útiles para los analistas de datos es `.groupby()`. Este método permite dividir los datos en grupos y a cada uno de estos aplicarles una función de agregación.

Veamos el siguiente ejemplo para entender este concepto mejor:

In [22]:
df = pd.read_excel(f"data/ejemplo_groupby.xlsx")
df

,animal,age,weight,length
0,hamster,1,7,8
1,alligator,9,13,6
2,hamster,4,8,9
3,cat,13,12,1
4,snake,14,11,8
5,cat,10,8,9
6,hamster,2,10,5
7,cat,4,14,6
8,cat,14,9,6
9,snake,7,11,6


In [24]:
df.animal.unique()

array(['hamster', 'alligator', 'cat', 'snake'], dtype=object)

Note que tenemos un `dataframe` con cuatro tipos de animales: 
- alligators (cocodrilos 🐊)
- cats (gatos 🐱)
- snakes (serpientes 🐍)
- hamsters (hamsters 🐹)

Cada una de las filas indican un chequeo en el veterinario donde se registra edad, peso y largo del animal. Por ende, usted como investigador quiere estudiar algunas estadísticas descriptivas por especie. Por ejemplo ¿Cuál es el peso promedio de cada especie?

In [25]:
# El primer paso es agrupar por animal
animal_groups = df.groupby("animal")

In [26]:
animal_groups

In [28]:
df

,animal,age,weight,length
0,hamster,1,7,8
1,alligator,9,13,6
2,hamster,4,8,9
3,cat,13,12,1
4,snake,14,11,8
5,cat,10,8,9
6,hamster,2,10,5
7,cat,4,14,6
8,cat,14,9,6
9,snake,7,11,6


In [27]:
# Veamos la conformación de cada uno de los grupos. ¿En qué filas aparece cada animal?
animal_groups.groups

{'alligator': [1, 13], 'cat': [3, 5, 7, 8, 12], 'hamster': [0, 2, 6, 10, 11], 'snake': [4, 9]}

Visualmente, lo que sucedió fue lo siguiente:

1. Se agrupa los valores únicos de la columna animal.
<center>
<img src = "./img/groupby1.jpg" width = "400">
</center>

2. La segmentación de cada grupo se vería de la siguiente manera
<center>
<img src = "./img/groupby2.jpg" width = "400">
</center>

3. Se le asignan las otras variables/columnas a cada grupo
<center>
<img src = "./img/groupby3.jpg" width = "400">
</center>

4. Se aplica la función agregadora `.mean()` sobre la columna `weight` de cada grupo.
<center>
<img src = "./img/groupby4.jpg" width = "400">
</center>


In [29]:
df

,animal,age,weight,length
0,hamster,1,7,8
1,alligator,9,13,6
2,hamster,4,8,9
3,cat,13,12,1
4,snake,14,11,8
5,cat,10,8,9
6,hamster,2,10,5
7,cat,4,14,6
8,cat,14,9,6
9,snake,7,11,6


In [30]:
df.groupby('animal')['weight'].mean()

animal
alligator    13.5
cat          10.4
hamster       9.0
snake        11.0
Name: weight, dtype: float64

#### Método .agg()
El método .agg() se puede utilizar después de aplicar un método .groupby() en pandas para realizar operaciones de agregación en los datos de cada grupo.

La sintaxis general de la función .groupby() es la siguiente:
```python
dataframe.groupby(columnas).agg(funciones)
```
Donde:
- dataframe: el DataFrame al que se aplicará la función `groupby()`.
- columnas: la(s) columna(s) que se utilizarán para agrupar los datos.
- funciones: la(s) operación(es) de agregación que se aplicarán a los datos agrupados.

Por ejemplo, para calcular la media, el máximo y el mínimo de las columnas de peso y longitud del DataFrame agrupado por la columna 'animal', se puede utilizar la siguiente sintaxis:

In [32]:
### Cuál es el peso y edad máxima por grupo de animal
df.groupby('animal')[['weight', 'age']].max()

,weight,age
animal,,
alligator,14,9
cat,14,14
hamster,10,14
snake,11,14


In [33]:
df.groupby("animal")[['weight','length']].agg(["min", "mean", "max"])

weight           length         
             min  mean max    min mean max
animal                                    
alligator     13  13.5  14      5  5.5   6
cat            8  10.4  14      1  5.2   9
hamster        7   9.0  10      3  6.0   9
snake         11  11.0  11      6  7.0   8

In [34]:
df.groupby("animal")['weight'].count()

animal
alligator    2
cat          5
hamster      5
snake        2
Name: weight, dtype: int64

In [35]:
df.groupby("animal").agg({'weight': ['mean', 'max'], 'length': 'std', 
                                     "age": lambda x: np.percentile(x, 50)})

weight        length      age
            mean max       std <lambda>
animal                                 
alligator   13.5  14  0.707107      8.0
cat         10.4  14  2.949576     10.0
hamster      9.0  10  2.449490      2.0
snake       11.0  11  1.414214     10.5

In [37]:
# Otra sintaxis, en vez de un diccionario, usar tuplas
# (nombre_columna,funcion)
df.groupby("animal").agg(peso_promedio = ("weight", 'mean'), 
                                   peso_maximo = ("weight", 'max'),
                                   edad_mediana = ("age", lambda x: np.percentile(x, 50)))

,peso_promedio,peso_maximo,edad_mediana
animal,,,
alligator,13.5,14,8.0
cat,10.4,14,10.0
hamster,9.0,10,2.0
snake,11.0,11,10.5


In [ ]:
## Siempre que querramos trabajar con la tabla colapsada como un dataframe hay que hacerle reset al index
df.groupby("animal").agg(peso_promedio = ("weight", 'mean'), 
                                   peso_maximo = ("weight", 'max'),
                                   edad_mediana = ("age", lambda x: np.percentile(x, 50))).reset_index()

,animal,peso_promedio,peso_maximo,edad_mediana
0,alligator,13.5,14,8.0
1,cat,10.4,14,10.0
2,hamster,9.0,10,2.0
3,snake,11.0,11,10.5


## Ejercicio

### Información de accidentes de tránsito en Bogotá en el 2016
A partir de la base de datos `info_accidentes.csv`, responda las siguientes preguntas:

1. ¿Cuántos muertos hay registrados en la base de datos?
2. ¿Cuántos heridos hay registrados en la base de datos?
3. ¿Cuantos accidentes hubo en la localidad de Santa Fe? ¿Cuántos muertos y cuántos heridos dejaron estos accidentes en total?
4. Explore la columna TipoTiempo, ¿cuál es la categoría en la que más ocurren accidentes?

In [39]:
df = pd.read_csv("./data/info_accidentes.csv")

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34931 entries, 0 to 34930
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Fecha             34931 non-null  object 
 1   GravedadNombre    34931 non-null  object 
 2   ClaseNombre       34931 non-null  object 
 3   ChoqueNombre      34931 non-null  object 
 4   ObjetoFijoCodigo  34931 non-null  object 
 5   ObjetoFijoNombre  34931 non-null  object 
 6   OtraClase         34931 non-null  object 
 7   NombreOtraClase   34931 non-null  object 
 8   Latitud           34931 non-null  float64
 9   Longitud          34931 non-null  float64
 10  Direccion         34931 non-null  object 
 11  TipoVia1          34931 non-null  object 
 12  NumeroVia1        34931 non-null  object 
 13  LetraVia1         34931 non-null  object 
 14  CardinalVia1      34931 non-null  object 
 15  TipoVia2          34931 non-null  object 
 16  NumeroVia2        34625 non-null  float6

In [41]:
df.head(2)

,Fecha,GravedadNombre,ClaseNombre,ChoqueNombre,ObjetoFijoCodigo,ObjetoFijoNombre,OtraClase,NombreOtraClase,Latitud,Longitud,...,TipoVia2,NumeroVia2,LetraVia2,CardinalVia2,Localidad,HoraOcurrencia,TipoDiseño,TipoTiempo,TotalMuertos,TotalHeridos
0,01/13/2016 12:00:00 AM,Con Heridos,Choque,Vehiculo,,,,,0.0,0.0,...,CL,83.0,,,ENGATIVA,12/31/1899 10:30:00 PM,Interseccion,Normal,0,2
1,01-12-16 0:00,Con Heridos,Atropello,,,,,,0.0,0.0,...,KR,7.0,,,USAQUEN,12/31/1899 03:40:00 PM,Interseccion,Normal,0,1


In [ ]:
### Cuátnos muertos hay en la base?
df['TotalMuertos'].sum()

np.int64(575)

In [43]:
## Cuántos heridos hay en la base??
df['TotalHeridos'].sum()

np.int64(14437)

In [44]:
df.columns

Index(['Fecha', 'GravedadNombre', 'ClaseNombre', 'ChoqueNombre',
       'ObjetoFijoCodigo', 'ObjetoFijoNombre', 'OtraClase', 'NombreOtraClase',
       'Latitud', 'Longitud', 'Direccion', 'TipoVia1', 'NumeroVia1',
       'LetraVia1', 'CardinalVia1', 'TipoVia2', 'NumeroVia2', 'LetraVia2',
       'CardinalVia2', 'Localidad', 'HoraOcurrencia', 'TipoDiseño',
       'TipoTiempo', 'TotalMuertos', 'TotalHeridos'],
      dtype='object')

In [46]:
### 
df['GravedadNombre'].value_counts(normalize=True)

GravedadNombre
Solo Daños     0.681859
Con Heridos    0.302253
Con Muertos    0.015888
Name: proportion, dtype: float64

In [47]:
##3. ¿Cuantos accidentes hubo en la localidad de Santa Fe? ¿Cuántos muertos y cuántos heridos dejaron estos accidentes en total?

In [48]:
df['Localidad'].unique()

array(['ENGATIVA', 'USAQUEN', 'SAN CRISTOBAL', 'RAFAEL URIBE URIBE',
       'KENNEDY', 'FONTIBON', 'ANTONIO NARIÑO', 'BARRIOS UNIDOS',
       'TUNJUELITO', 'LOS MARTIRES', 'TEUSAQUILLO', 'CIUDAD BOLIVAR',
       'SUBA', 'CHAPINERO', 'USME', 'SANTA FE', 'BOSA', 'PUENTE ARANDA',
       'CANDELARIA'], dtype=object)

In [49]:
santa_fe=df.loc[df['Localidad']=='SANTA FE']
len(santa_fe)

967

In [51]:
df.head(2)

,Fecha,GravedadNombre,ClaseNombre,ChoqueNombre,ObjetoFijoCodigo,ObjetoFijoNombre,OtraClase,NombreOtraClase,Latitud,Longitud,...,TipoVia2,NumeroVia2,LetraVia2,CardinalVia2,Localidad,HoraOcurrencia,TipoDiseño,TipoTiempo,TotalMuertos,TotalHeridos
0,01/13/2016 12:00:00 AM,Con Heridos,Choque,Vehiculo,,,,,0.0,0.0,...,CL,83.0,,,ENGATIVA,12/31/1899 10:30:00 PM,Interseccion,Normal,0,2
1,01-12-16 0:00,Con Heridos,Atropello,,,,,,0.0,0.0,...,KR,7.0,,,USAQUEN,12/31/1899 03:40:00 PM,Interseccion,Normal,0,1


In [50]:
## Número de muertos y heridos en Santa Fe
santa_fe[['TotalMuertos','TotalHeridos']].sum()

TotalMuertos     16
TotalHeridos    439
dtype: int64

In [55]:
### encontremos la localidad con el mayor número de heridos en los accidentes
agrupado_localidad= df.groupby('Localidad')['TotalHeridos'].sum().reset_index()

agrupado_localidad.sort_values(by='TotalHeridos', ascending=False)

,Localidad,TotalHeridos
8,KENNEDY,1939
14,SUBA,1334
6,ENGATIVA,1242
10,PUENTE ARANDA,1084
5,CIUDAD BOLIVAR,883
17,USAQUEN,882
2,BOSA,855
7,FONTIBON,832
4,CHAPINERO,628
1,BARRIOS UNIDOS,627


In [56]:
### encontremos la localidad con el mayor número de heridos en los accidentes
agrupado_localidad= df.groupby('Localidad')['TotalMuertos'].sum().reset_index()

agrupado_localidad.sort_values(by='TotalMuertos', ascending=False)

,Localidad,TotalMuertos
8,KENNEDY,78
6,ENGATIVA,63
14,SUBA,43
10,PUENTE ARANDA,42
7,FONTIBON,39
5,CIUDAD BOLIVAR,39
17,USAQUEN,35
2,BOSA,33
16,TUNJUELITO,26
11,RAFAEL URIBE URIBE,25


In [61]:
agrupado_localidad= df.groupby('Localidad').agg(total_muertos=('TotalMuertos','sum'), total_accidentes =('TotalMuertos','count')).reset_index()
agrupado_localidad['letalidad']= agrupado_localidad['total_muertos']/agrupado_localidad['total_accidentes'] 
agrupado_localidad.sort_values(by='letalidad', ascending=False)

,Localidad,total_muertos,total_accidentes,letalidad
18,USME,21,654,0.032110
0,ANTONIO NARIÑO,19,666,0.028529
5,CIUDAD BOLIVAR,39,1389,0.028078
16,TUNJUELITO,26,932,0.027897
11,RAFAEL URIBE URIBE,25,933,0.026795
12,SAN CRISTOBAL,19,863,0.022016
2,BOSA,33,1524,0.021654
8,KENNEDY,78,4009,0.019456
6,ENGATIVA,63,3487,0.018067
10,PUENTE ARANDA,42,2409,0.017435


In [62]:
agrupado_localidad= df.groupby('Localidad').agg(total_muertos=('TotalHeridos','sum'), total_accidentes =('TotalHeridos','count')).reset_index()
agrupado_localidad['letalidad']= agrupado_localidad['total_muertos']/agrupado_localidad['total_accidentes'] 
agrupado_localidad.sort_values(by='letalidad', ascending=False)

,Localidad,total_muertos,total_accidentes,letalidad
12,SAN CRISTOBAL,565,863,0.654693
11,RAFAEL URIBE URIBE,600,933,0.643087
5,CIUDAD BOLIVAR,883,1389,0.635709
16,TUNJUELITO,578,932,0.620172
2,BOSA,855,1524,0.561024
18,USME,361,654,0.551988
0,ANTONIO NARIÑO,361,666,0.542042
8,KENNEDY,1939,4009,0.483662
9,LOS MARTIRES,546,1176,0.464286
13,SANTA FE,439,967,0.453981


In [66]:
## Qué pasa con la columna tipotiempo
df.groupby('TipoTiempo')['TotalHeridos'].sum().reset_index()

,TipoTiempo,TotalHeridos
0,,4
1,Lluvia,390
2,Lluvia/Lluvia,3
3,Lluvia/Normal,1
4,Niebla,15
5,Normal,13995
6,Normal/Lluvia,0
7,Normal/Normal,6
8,Viento,23
9,Viento/Normal,0


In [68]:
df.head(2)

,Fecha,GravedadNombre,ClaseNombre,ChoqueNombre,ObjetoFijoCodigo,ObjetoFijoNombre,OtraClase,NombreOtraClase,Latitud,Longitud,...,TipoVia2,NumeroVia2,LetraVia2,CardinalVia2,Localidad,HoraOcurrencia,TipoDiseño,TipoTiempo,TotalMuertos,TotalHeridos
0,01/13/2016 12:00:00 AM,Con Heridos,Choque,Vehiculo,,,,,0.0,0.0,...,CL,83.0,,,ENGATIVA,12/31/1899 10:30:00 PM,Interseccion,Normal,0,2
1,01-12-16 0:00,Con Heridos,Atropello,,,,,,0.0,0.0,...,KR,7.0,,,USAQUEN,12/31/1899 03:40:00 PM,Interseccion,Normal,0,1
